In [ ]:
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# Список файлов
files = {
    'qwen_tqa_cut.csv': 'Qwen_TruthfulQA',
    'qwen_slava_cut.csv': 'Qwen_Slava',
    'yandex_gpt_slava_answered.csv': 'YandexGPT_Slava',
    'yandex_gpt_tqa_answered.csv': 'YandexGPT_TruthfulQA',
    'gpt_slava_annotated.csv': 'ChatGPT_Slava',
    'gpt_tqa_answered.csv': 'ChatGPT_TruthfulQA',
    'giga_slava_annotated.csv': 'GigaChat_Slava',
    'giga_tqa_answered.csv': 'GigaChat_TruthfulQA'
}

feature_metrics = ['Semantic Entropy', 'EigenScore', 'SAR Score', 'NumSets', 'Self Confidence', 'Verbalized Uncertainty']

all_data = []
for file, name in files.items():
    try:
        df = pd.read_csv(file, encoding='utf-8')
        df['source'] = name
        all_data.append(df)
        print(f"✅ Loaded {file}: {len(df)} rows")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

df_all = pd.concat(all_data, ignore_index=True)
print(f"\n📊 Total: {len(df_all)} rows")

available_features = [col for col in feature_metrics if col in df_all.columns]
print(f"✅ Available features: {available_features}")

if 'ground_truth' in df_all.columns:
    df_all['target'] = df_all['ground_truth']
    print(f"Target distribution: {df_all['target'].value_counts().to_dict()}")
else:
    print("❌ ground_truth column not found!")

df_all['model'] = df_all['source'].apply(lambda x: x.split('_')[0])
df_all['dataset'] = df_all['source'].apply(lambda x: x.split('_')[1])

✅ Loaded qwen_tqa_cut.csv: 790 rows
✅ Loaded qwen_slava_cut.csv: 500 rows
✅ Loaded yandex_gpt_slava_answered.csv: 481 rows
✅ Loaded yandex_gpt_tqa_answered.csv: 755 rows
✅ Loaded gpt_slava_annotated.csv: 500 rows
✅ Loaded gpt_tqa_answered.csv: 785 rows
✅ Loaded giga_slava_annotated.csv: 500 rows
✅ Loaded giga_tqa_answered.csv: 761 rows

📊 Total: 5072 rows
✅ Available features: ['Semantic Entropy', 'EigenScore', 'SAR Score', 'NumSets', 'Self Confidence', 'Verbalized Uncertainty']
Target distribution: {0: 4014, 1: 1058}


In [ ]:
# МОДЕЛИ И ГИПЕРПАРАМЕТРЫ

# 1. Logistic Regression
logreg_params = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs'],
    'class_weight': ['balanced', None]
}

# 2. Random Forest
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None]
}

# 3. Gradient Boosting
gb_params = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

# 4. SVM
svm_params = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}

# 5. MLP (Neural Network)
mlp_params = {
    'hidden_layer_sizes': [(25,), (50,), (50, 25)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['constant', 'adaptive']
}

# 6. XGBoost
xgb_params = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'eval_metric': ['logloss']
}

# 7. CatBoost
catboost_params = {
    'iterations': [50, 100],
    'depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'border_count': [32, 64]
}

models_with_params = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), logreg_params),
    'Random Forest': (RandomForestClassifier(random_state=42), rf_params),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=42), gb_params),
    'SVM': (SVC(probability=True, random_state=42), svm_params),
    'MLP': (MLPClassifier(max_iter=500, random_state=42), mlp_params),
    'XGBoost': (XGBClassifier(random_state=42, use_label_encoder=False, verbosity=0), xgb_params),
    'CatBoost': (CatBoostClassifier(random_state=42, verbose=0), catboost_params)
}

In [ ]:

#   ОЦЕНКА С ОПТИМИЗАЦИЕЙ ПО RECALL

def evaluate_with_recall_optimization(df, dataset_name, features):

    X = df[features].fillna(df[features].mean())
    y = df['target']

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    results = []
    for name, (model, params) in models_with_params.items():
        print(f"  Optimizing {name} on {dataset_name} (scoring=recall)...")

        # GridSearchCV с оптимизацией по RECALL
        grid_search = GridSearchCV(
            model, params,
            cv=3,
            scoring='recall',
            n_jobs=-1,
            verbose=0
        )
        grid_search.fit(X, y)
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        f1_scores = []
        precision_scores = []
        recall_scores = []
        roc_scores = []

        for train_idx, test_idx in cv.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            model_clone = best_model.__class__(**best_model.get_params())
            model_clone.fit(X_train, y_train)

            if hasattr(model_clone, "predict_proba"):
                y_proba = model_clone.predict_proba(X_test)[:, 1]
                y_pred = (y_proba > 0.5).astype(int)
                roc_scores.append(roc_auc_score(y_test, y_proba))
            else:
                y_pred = model_clone.predict(X_test)
                roc_scores.append(0.5)

            f1_scores.append(f1_score(y_test, y_pred, zero_division=0))
            precision_scores.append(precision_score(y_test, y_pred, zero_division=0))
            recall_scores.append(recall_score(y_test, y_pred, zero_division=0))

        results.append({
            'dataset': dataset_name,
            'model': name,
            'best_params': str(best_params),
            'f1': np.mean(f1_scores),
            'f1_std': np.std(f1_scores),
            'precision': np.mean(precision_scores),
            'precision_std': np.std(precision_scores),
            'recall': np.mean(recall_scores),
            'recall_std': np.std(recall_scores),
            'roc_auc': np.mean(roc_scores),
            'roc_auc_std': np.std(roc_scores)
        })

    return pd.DataFrame(results)

In [ ]:

# ЗАПУСК НА ВСЕХ СРЕЗАХ

all_results = []

print("="*80)
print("1. ОТДЕЛЬНЫЕ ФАЙЛЫ (8 шт)")
print("="*80)

for file, name in files.items():
    try:
        df = pd.read_csv(file, encoding='utf-8')
        if 'ground_truth' in df.columns:
            df['target'] = df['ground_truth']
        print(f"\n📁 Processing {name} ({len(df)} samples)...")
        results = evaluate_with_recall_optimization(df, name, available_features)
        all_results.append(results)
        print(f"✅ {name} completed")
    except Exception as e:
        print(f"❌ Error on {file}: {e}")

print("\n" + "="*80)
print("2. ПО ДАТАСЕТАМ (TruthfulQA vs Slava)")
print("="*80)

for dataset_name in ['TruthfulQA', 'Slava']:
    df_subset = df_all[df_all['dataset'] == dataset_name]
    if len(df_subset) > 0:
        print(f"\n📁 Processing ALL_{dataset_name} ({len(df_subset)} samples)...")
        results = evaluate_with_recall_optimization(df_subset, f"ALL_{dataset_name}", available_features)
        all_results.append(results)
        print(f"✅ ALL_{dataset_name} completed")

print("\n" + "="*80)
print("3. ПО МОДЕЛЯМ")
print("="*80)

for model_name in ['Qwen', 'YandexGPT', 'ChatGPT', 'GigaChat']:
    df_subset = df_all[df_all['model'] == model_name]
    if len(df_subset) > 0:
        print(f"\n📁 Processing ALL_{model_name} ({len(df_subset)} samples)...")
        results = evaluate_with_recall_optimization(df_subset, f"ALL_{model_name}", available_features)
        all_results.append(results)
        print(f"✅ ALL_{model_name} completed")

print("\n" + "="*80)
print("4. ПОЛНЫЙ ДАТАСЕТ")
print("="*80)

print(f"\n📁 Processing FULL_DATASET ({len(df_all)} samples)...")
results = evaluate_with_recall_optimization(df_all, "FULL_DATASET", available_features)
all_results.append(results)
print("✅ FULL_DATASET completed")

# Объединение
df_all_results = pd.concat(all_results, ignore_index=True)
print(f"\n📊 Total evaluations: {len(df_all_results)}")

1. ОТДЕЛЬНЫЕ ФАЙЛЫ (8 шт)

📁 Processing Qwen_TruthfulQA (790 samples)...
  Optimizing Logistic Regression on Qwen_TruthfulQA (scoring=recall)...
  Optimizing Random Forest on Qwen_TruthfulQA (scoring=recall)...
  Optimizing Gradient Boosting on Qwen_TruthfulQA (scoring=recall)...
  Optimizing SVM on Qwen_TruthfulQA (scoring=recall)...
  Optimizing MLP on Qwen_TruthfulQA (scoring=recall)...
  Optimizing XGBoost on Qwen_TruthfulQA (scoring=recall)...
  Optimizing CatBoost on Qwen_TruthfulQA (scoring=recall)...
✅ Qwen_TruthfulQA completed

📁 Processing Qwen_Slava (500 samples)...
  Optimizing Logistic Regression on Qwen_Slava (scoring=recall)...
  Optimizing Random Forest on Qwen_Slava (scoring=recall)...
  Optimizing Gradient Boosting on Qwen_Slava (scoring=recall)...
  Optimizing SVM on Qwen_Slava (scoring=recall)...
  Optimizing MLP on Qwen_Slava (scoring=recall)...
  Optimizing XGBoost on Qwen_Slava (scoring=recall)...
  Optimizing CatBoost on Qwen_Slava (scoring=recall)...
✅ Qwen_Sla

In [ ]:
#Resutls

print("="*80)
print("ТАБЛИЦЫ ПО КАЖДОМУ СРЕЗУ (оптимизация по RECALL)")
print("="*80)

def save_table(df, slice_names, filename, title):
    df_subset = df[df['dataset'].isin(slice_names)]
    if df_subset.empty:
        return None
    table = df_subset[['dataset', 'model', 'recall', 'f1', 'precision', 'roc_auc']].copy()
    table = table.round(4)
    table = table.sort_values(['dataset', 'recall'], ascending=[True, False])
    print(f"\n{title}")
    display(table)
    table.to_csv(filename, index=False)
    return table

# Таблица 1: Отдельные файлы
individual_names = list(files.values())
table1 = save_table(df_all_results, individual_names, 'table1_individual.csv', "📊 ТАБЛИЦА 1: ОТДЕЛЬНЫЕ ФАЙЛЫ (оптимизация по RECALL)")

# Таблица 2: По датасетам
dataset_names = [d for d in df_all_results['dataset'].unique() if d.startswith('ALL_TruthfulQA') or d.startswith('ALL_Slava')]
table2 = save_table(df_all_results, dataset_names, 'table2_by_dataset.csv', "📊 ТАБЛИЦА 2: ПО ДАТАСЕТАМ (оптимизация по RECALL)")

# Таблица 3: По моделям
model_names = [d for d in df_all_results['dataset'].unique() if d.startswith('ALL_Qwen') or d.startswith('ALL_YandexGPT') or d.startswith('ALL_ChatGPT') or d.startswith('ALL_GigaChat')]
table3 = save_table(df_all_results, model_names, 'table3_by_model.csv', "📊 ТАБЛИЦА 3: ПО МОДЕЛЯМ (оптимизация по RECALL)")

# Таблица 4: Полный датасет
table4 = save_table(df_all_results, ['FULL_DATASET'], 'table4_full.csv', "📊 ТАБЛИЦА 4: ПОЛНЫЙ ДАТАСЕТ (оптимизация по RECALL)")

ТАБЛИЦЫ ПО КАЖДОМУ СРЕЗУ (оптимизация по RECALL)

📊 ТАБЛИЦА 1: ОТДЕЛЬНЫЕ ФАЙЛЫ (оптимизация по RECALL)


,dataset,model,recall,f1,precision,roc_auc
28,ChatGPT_Slava,Logistic Regression,0.7000,0.3472,0.2321,0.7852
29,ChatGPT_Slava,Random Forest,0.1500,0.1564,0.1697,0.6716
30,ChatGPT_Slava,Gradient Boosting,0.1500,0.1831,0.2386,0.6793
33,ChatGPT_Slava,XGBoost,0.0750,0.1133,0.2500,0.7385
31,ChatGPT_Slava,SVM,0.0500,0.0844,0.3000,0.7874
34,ChatGPT_Slava,CatBoost,0.0500,0.0800,0.2000,0.7640
32,ChatGPT_Slava,MLP,0.0000,0.0000,0.0000,0.7931
35,ChatGPT_TruthfulQA,Logistic Regression,0.5381,0.4291,0.3571,0.7882
36,ChatGPT_TruthfulQA,Random Forest,0.4629,0.4536,0.4487,0.7957
37,ChatGPT_TruthfulQA,Gradient Boosting,0.3848,0.4190,0.4667,0.7889



📊 ТАБЛИЦА 2: ПО ДАТАСЕТАМ (оптимизация по RECALL)


,dataset,model,recall,f1,precision,roc_auc
63,ALL_Slava,Logistic Regression,0.7293,0.5533,0.4466,0.7931
64,ALL_Slava,Random Forest,0.5676,0.5262,0.4931,0.7909
65,ALL_Slava,Gradient Boosting,0.4010,0.4686,0.5670,0.7925
68,ALL_Slava,XGBoost,0.3906,0.4584,0.5617,0.7921
69,ALL_Slava,CatBoost,0.3854,0.4614,0.5827,0.8152
67,ALL_Slava,MLP,0.3336,0.3974,0.5496,0.7966
66,ALL_Slava,SVM,0.2186,0.3157,0.5732,0.8005
56,ALL_TruthfulQA,Logistic Regression,0.6188,0.4615,0.3685,0.6882
57,ALL_TruthfulQA,Random Forest,0.5163,0.4653,0.4243,0.7282
61,ALL_TruthfulQA,XGBoost,0.2314,0.3049,0.4575,0.7203



📊 ТАБЛИЦА 3: ПО МОДЕЛЯМ (оптимизация по RECALL)


,dataset,model,recall,f1,precision,roc_auc
84,ALL_ChatGPT,Logistic Regression,0.5901,0.4064,0.3110,0.7341
85,ALL_ChatGPT,Random Forest,0.3067,0.3200,0.3475,0.7174
86,ALL_ChatGPT,Gradient Boosting,0.2022,0.2669,0.4205,0.7332
89,ALL_ChatGPT,XGBoost,0.1815,0.2468,0.4038,0.7118
90,ALL_ChatGPT,CatBoost,0.1813,0.2690,0.6656,0.7846
88,ALL_ChatGPT,MLP,0.1810,0.2717,0.5722,0.7464
87,ALL_ChatGPT,SVM,0.0069,0.0129,0.1000,0.7719
91,ALL_GigaChat,Logistic Regression,0.5746,0.2817,0.1868,0.5871
92,ALL_GigaChat,Random Forest,0.1319,0.1469,0.1796,0.5203
93,ALL_GigaChat,Gradient Boosting,0.0514,0.0803,0.1889,0.5316



📊 ТАБЛИЦА 4: ПОЛНЫЙ ДАТАСЕТ (оптимизация по RECALL)


,dataset,model,recall,f1,precision,roc_auc
98,FULL_DATASET,Logistic Regression,0.6976,0.4840,0.3710,0.7288
99,FULL_DATASET,Random Forest,0.5521,0.4743,0.4169,0.7362
103,FULL_DATASET,XGBoost,0.2599,0.3321,0.4634,0.7377
100,FULL_DATASET,Gradient Boosting,0.2174,0.2968,0.4732,0.7430
104,FULL_DATASET,CatBoost,0.1465,0.2281,0.5397,0.7577
102,FULL_DATASET,MLP,0.0964,0.1573,0.5285,0.7352
101,FULL_DATASET,SVM,0.0482,0.0876,0.5183,0.7294
